**LOGISTIC REGRESSION CLASSIFICATION MODEL**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report
import joblib


In [ ]:
# apply rolling window to each setup
def apply_rolling_window(setup_df):

    window_size = 30

    setup_df['MAR_mean'] = setup_df['MAR'].rolling(window_size).mean()
    setup_df['EAR_mean'] = setup_df['EAR'].rolling(window_size).mean()
    setup_df['MAR_std'] = setup_df['MAR'].rolling(window_size).std()
    setup_df['EAR_std'] = setup_df['EAR'].rolling(window_size).std()
    setup_df = setup_df.dropna()

    return setup_df

# encode labels to numeric labels
def label_class(setup_df):
    le = LabelEncoder()
    setup_df['class'] = le.fit_transform(setup_df['class'])
    
    return setup_df

In [ ]:
# dataset splits
train_setups = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
val_setups = [11, 12, 13]
test_setups = [14, 15]

In [ ]:
train_dfs = []
val_dfs = []
test_dfs = []

for setup in train_setups:
    setup_df = pd.read_csv(f'..\\..\\CSV Dataset\\landmarks_setup_{setup}.csv')
    print('Processing setup ' + str(setup) + '(' + str(setup_df.shape[0]) + ' -> ', end="")
    setup_df = apply_rolling_window(setup_df)
    setup_df = label_class(setup_df)
    train_dfs.append(setup_df)
    print(setup_df.shape[0].__str__() + ') frames')

for setup in val_setups:
    setup_df = pd.read_csv(f'..\\..\\CSV Dataset\\landmarks_setup_{setup}.csv')
    print('Processing setup ' + str(setup) + '(' + str(setup_df.shape[0]) + ' -> ', end="")
    setup_df = apply_rolling_window(setup_df)
    setup_df = label_class(setup_df)
    val_dfs.append(setup_df)
    print(setup_df.shape[0].__str__() + ') frames')

for setup in test_setups:
    setup_df = pd.read_csv(f'..\\..\\CSV Dataset\\landmarks_setup_{setup}.csv')
    print('Processing setup ' + str(setup) + '(' + str(setup_df.shape[0]) + ' -> ', end="")
    setup_df = apply_rolling_window(setup_df)
    setup_df = label_class(setup_df)
    test_dfs.append(setup_df)
    print(setup_df.shape[0].__str__() + ') frames')

train_df = pd.concat(train_dfs, ignore_index=True)
val_df = pd.concat(val_dfs, ignore_index=True)
test_df = pd.concat(test_dfs, ignore_index=True)

In [ ]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

**Class Distribution Comparison**

In [ ]:
alert = train_df[train_df['class'] == 0]
drowsy = train_df[train_df['class'] == 1]
microsleep = train_df[train_df['class'] == 2]

# undersample alert to double the count of drowsy class
alert_downsampled_1 = resample(alert, n_samples=len(drowsy)*2, random_state=42)

train_balanced_1 = pd.concat([alert_downsampled_1, drowsy, microsleep])
train_balanced_1 = train_balanced_1.sample(frac=1, random_state=42).reset_index(drop=True)

# undersamle alter to equal the count of drowsy class
alert_downsampled_2 = resample(alert, n_samples=len(drowsy), random_state=42)

train_balanced_2 = pd.concat([alert_downsampled_2, drowsy, microsleep])
train_balanced_2 = train_balanced_2.sample(frac=1, random_state=42).reset_index(drop=True)

# SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(train_df.drop(['class'], axis=1), train_df['class'])
train_balanced_3 = X_resampled.copy()
train_balanced_3['class'] = y_resampled
train_balanced_3 = train_balanced_3.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
print("Initial Distributions")
print(train_df['class'].value_counts())

print("\nUndersampling with alert with 2x drowsy")
print(train_balanced_1['class'].value_counts())

print("\nUndersampling with alert with equal count to drowsy")
print(train_balanced_2['class'].value_counts())

print("\nSMOTE")
print(train_balanced_3['class'].value_counts())

In [ ]:
positions = [0, 1, 2]
labels = ['Awake', 'Drowsy', 'Microsleep']

fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].bar(train_df['class'].value_counts().index, train_df['class'].value_counts().values)
ax[0].set_title('Unbalanced')
ax[1].bar(train_balanced_1['class'].value_counts().index, train_balanced_1['class'].value_counts().values)
ax[1].set_title('2x Drowsy')
ax[2].bar(train_balanced_2['class'].value_counts().index, train_balanced_2['class'].value_counts().values)
ax[2].set_title('Equal Drowsy')
ax[3].bar(train_balanced_3['class'].value_counts().index, train_balanced_3['class'].value_counts().values)
ax[3].set_title('SMOTE')

for a in ax:
    a.set_xticks(positions)
    a.set_xticklabels(labels)

**Train Logistic Regression Model on all Class Balancing Strategies**

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, classification_report

strategies = {
    'Unbalanced': train_df,
    '2x Alert': train_balanced_1,
    '1x Alert': train_balanced_2,
    'SMOTE': train_balanced_3
}

X_val = val_df.drop(['class'], axis=1)
y_val = val_df['class']

rows = []

for name, strategy in strategies.items():
    X_train = strategy.drop(['class'], axis=1)
    y_train = strategy['class']

    model = LogisticRegression(solver='lbfgs', max_iter=1000, class_weight='balanced', random_state=42, multi_class='multinomial')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    f1_per_class = f1_score(y_val, y_pred, average=None)

    rows.append({
        'Strategy': name,
        'Alert F1': round(f1_per_class[0], 4),
        'Drowsy F1': round(f1_per_class[1], 4),
        'Microsleep F1': round(f1_per_class[2], 4),
        'Macro F1': round(f1_score(y_val, y_pred, average='macro'), 4),
        'Accuracy': round(accuracy_score(y_val, y_pred), 4)
    })

results_df = pd.DataFrame(rows).set_index('Strategy')
print(results_df)

In [ ]:
# use the best strategy (unbalanced)
X_train = train_df.drop(['class'], axis=1)
y_train = train_df['class']

X_test = test_df.drop(['class'], axis=1)
y_test = test_df['class']

model = LogisticRegression(solver='lbfgs', max_iter=1000, class_weight='balanced', random_state=42, multi_class='multinomial')
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

In [ ]:

joblib.dump(model, "LogisticRegressionModel.joblib")